# Download and install Scenic

In [1]:
!rm -rf * # remove, recursively and forcefully
!rm -rf .config  # remove the .config directory
!rm -rf .git
!git clone https://github.com/google-research/scenic.git .
!python -m pip install -q .
# !python -m pip install -r ./scenic/projects/mbt/requirements.txt

Cloning into '.'...
remote: Enumerating objects: 11519, done.
remote: Counting objects: 100% (534/534), done.
remote: Compressing objects: 100% (270/270), done.
remote: Total 11519 (delta 418), reused 269 (delta 264), pack-reused 10985 (from 3)
Receiving objects: 100% (11519/11519), 68.84 MiB | 21.09 MiB/s, done.
Resolving deltas: 100% (7751/7751), done.
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 7.9 MB/s eta 0:00:00


# Train [a simple feedforward network on mnist](https://github.com/google-research/scenic/blob/main/scenic/projects/baselines/configs/mnist/mnist_config.py)

In [7]:
import jax
import jax.numpy as jnp
import ml_collections
from scenic.projects.mbt import model as mbt_model

# 1. Define the parameters explicitly to feed the MBT layer blueprint
config = ml_collections.ConfigDict()
config.model = ml_collections.ConfigDict()
config.model.use_bottleneck = True
config.model.test_with_bottlenecks = True
config.model.share_encoder = False
config.model.n_bottlenecks = 4
config.model.fusion_layer = 8
config.model.num_layers = 12
config.model.hidden_size = 768
config.model.num_heads = 12
config.model.mlp_dim = 3072

# --- THE FIX IS HERE ---
# Changed from 'gap' to 'token' to bypass the UnboundLocalError bug in Google's code
config.model.classifier = 'token'
# -----------------------

config.model.dropout_rate = 0.0
config.model.attention_dropout_rate = 0.0
config.model_dtype_str = 'float32'

config.model.patches = ml_collections.ConfigDict()
config.model.patches.size = [16, 16, 2]

config.model.temporal_encoding_config = ml_collections.ConfigDict()
config.model.temporal_encoding_config.method = '3d_conv'

config.model.attention_config = ml_collections.ConfigDict()
config.model.attention_config.type = 'spacetime'

# 2. Instantiate the core MBT architecture class directly
model = mbt_model.MBT(
    num_classes=527,
    modality_fusion=('spectrogram', 'rgb'),
    fusion_layer=config.model.fusion_layer,
    use_bottleneck=config.model.use_bottleneck,
    test_with_bottlenecks=config.model.test_with_bottlenecks,
    n_bottlenecks=config.model.n_bottlenecks,
    share_encoder=config.model.share_encoder,
    mlp_dim=config.model.mlp_dim,
    num_layers=config.model.num_layers,
    num_heads=config.model.num_heads,
    patches=config.model.patches,
    hidden_size=config.model.hidden_size,
    temporal_encoding_config=config.model.temporal_encoding_config,
    attention_config=config.model.attention_config,
    classifier=config.model.classifier,
    dropout_rate=config.model.dropout_rate,
    attention_dropout_rate=config.model.attention_dropout_rate
)

# 3. Formulate the dictionary input payload
dummy_video = jnp.ones((1, 32, 224, 224, 3))
dummy_audio = jnp.ones((1, 100, 128))[..., jnp.newaxis]

input_payload = {
    'rgb': dummy_video,
    'spectrogram': dummy_audio
}

# ... (Keep all your config and dummy_video/dummy_audio creation the same)

# 4. Initialize random variables
rng = jax.random.PRNGKey(42)
print("Compiling network structures...")

# FIX: Pass a dynamically created dictionary so the original tensors are untouched
variables = model.init(rng, {
    'rgb': dummy_video,
    'spectrogram': dummy_audio
}, train=False)

# 5. Execute inference pass
# FIX: Pass ANOTHER fresh dictionary for the actual inference run
logits = model.apply(variables, {
    'rgb': dummy_video,
    'spectrogram': dummy_audio
}, train=False)

print("--- Execution Complete ---")
print(f"Logits vector shape out: {logits.shape}")

Compiling network structures...
--- Execution Complete ---
Logits vector shape out: (1, 527)


In [16]:
!ls scenic/projects/mbt/configs/

audioset  __init__.py
